# Paper Figures — Tier A

Generates the strongest main-text figures for the MALCA dipper paper from the July 1 review run.

Outputs are written to `output/notebooks/paper_figures/` (PDF + PNG). SED-based panels require the marked-dipper SED sidecars produced by `july1_marked_dipper_seds.ipynb` or `malca sed-excess`.

The depth–timescale figure (A4) auto-discovers injection-recovery output and overlays the 2D completeness map (fractional depth vs. timescale, marginalized over magnitude). It prefers `injection_results.parquet` and falls back to `efficiency_cube.npz` if needed.

## Parameters

In [ ]:
%matplotlib inline

from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from malca.io.notebook_paths import find_repo_root

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from malca.plotting.notebook_display import reload_plotting_modules, show_figure

reload_plotting_modules()

from malca.plotting.lightcurve_publication import apply_publication_rcparams
from malca.plotting.paper_figures import (
    PaperFigureContext,
    default_context,
    generate_tier_a,
    load_dippers,
    load_survey_candidates,
    plot_amplitude_vs_median_g,
    plot_dip_depth_timescale,
    plot_error_vs_median_g,
    plot_extinction_corrected_cmd,
    plot_ir_excess_vs_bprp,
    plot_ks_w2_w4_dered,
    plot_mq_diagram,
    plot_representative_seds,
    plot_sed_with_residuals,
    run_existing_cmd_script,
)

RUN_NAME = "dat3-full-extended_2026-07-01-v4"
RUN_ROOT = REPO_ROOT / "output" / "runs" / RUN_NAME
REVIEW_DB = RUN_ROOT / "review" / "review.db"
OUTPUT_DIR = REPO_ROOT / "output" / "notebooks" / "paper_figures"

INJECTION_RECOVERY_TEXT = os.environ.get("MALCA_INJECTION_RESULTS", os.environ.get("MALCA_EFFICIENCY_CUBE", "")).strip()
INJECTION_RECOVERY = Path(INJECTION_RECOVERY_TEXT).expanduser() if INJECTION_RECOVERY_TEXT else None

EXPORT_PDF = True
EXPORT_PNG = True
RUN_FULL_CMD_SCRIPT = False  # set True for the full background-density CMD from scripts/plot_march18_review_cmd.py
SED_RESIDUAL_CANDIDATE = None  # e.g. 'stv_111669273145'; None picks best reduced_chi2

ctx = PaperFigureContext(
    repo_root=REPO_ROOT,
    run_root=RUN_ROOT,
    review_db=REVIEW_DB,
    output_dir=OUTPUT_DIR,
    injection_recovery_path=INJECTION_RECOVERY,
    export_pdf=EXPORT_PDF,
    export_png=EXPORT_PNG,
    show_inline=True,
)
ctx.output_dir.mkdir(parents=True, exist_ok=True)

apply_publication_rcparams(plt)
print(f"Review DB: {REVIEW_DB}")
print(f"Output dir: {OUTPUT_DIR}")
if ctx.injection_recovery_path:
    print(f"Injection recovery: {ctx.injection_recovery_path}")
else:
    print(
        "Injection recovery: (none found — A4 depth–timescale plot omits completeness overlay; "
        "run `malca dip-injection --manifest "
        f"{RUN_ROOT / 'results' / 'external_lc_manifest.parquet'}`)"
    )

## Load data

In [ ]:
survey = load_survey_candidates(REVIEW_DB)
dippers = load_dippers(REVIEW_DB)
display(Markdown(f"Survey candidates: **{len(survey):,}** | Reviewed dippers: **{len(dippers):,}**"))

## A1 — Variability amplitude vs. median $g$

In [ ]:
show_figure(*plot_amplitude_vs_median_g(ctx, survey))

## A2 — Photometric error vs. median $g$

In [ ]:
show_figure(*plot_error_vs_median_g(ctx, survey))

## A3 — $M$ vs. $Q$ morphology diagram

In [ ]:
show_figure(*plot_mq_diagram(ctx, survey))

## A4 — Dip amplitude vs. dip timescale

Completeness overlay uses the 2D detection-efficiency map (depth × timescale) from injection-recovery results. Auto-discovery prefers `injection_results.parquet`; override with `MALCA_INJECTION_RESULTS` if needed.

In [ ]:
show_figure(*plot_dip_depth_timescale(ctx, dippers, overlay_efficiency=True))

## A5 — IR excess vs. $G_\mathrm{BP}-G_\mathrm{RP}$ (dippers vs. null controls)

In [ ]:
show_figure(*plot_ir_excess_vs_bprp(ctx, dippers))

## A6 — Extinction-corrected $K_s-W2$ vs. $K_s-W4$

In [ ]:
show_figure(*plot_ks_w2_w4_dered(ctx, dippers))

## A7 — Extinction-corrected Gaia CMD

This cell uses a lightweight in-notebook CMD. For the full Gaia background + MIST isochrone figure, set `RUN_FULL_CMD_SCRIPT = True` in the parameters cell.

In [ ]:
show_figure(*plot_extinction_corrected_cmd(ctx, dippers))

In [ ]:
if RUN_FULL_CMD_SCRIPT:
    cmd_paths = run_existing_cmd_script(ctx)
    display(cmd_paths)

## A8 — Representative SED decompositions

In [ ]:
show_figure(*plot_representative_seds(ctx, n_per_class=1))

## A9 — SED fit with residual panel

In [ ]:
show_figure(*plot_sed_with_residuals(ctx, candidate_id=SED_RESIDUAL_CANDIDATE))

## Batch export (all Tier A figures)

In [ ]:
outputs = generate_tier_a(ctx, include_cmd_script=RUN_FULL_CMD_SCRIPT)
for name, (paths, _fig) in outputs.items():
    print(name, "->", ", ".join(str(p) for p in paths))